<a href="https://colab.research.google.com/github/jeffheaton/t81_558_deep_learning/blob/master/assignments/assignment_yourname_t81_558_class4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# T81-558: Anwendungen tiefer neuronaler Netzwerke
* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

**Aufgabe Modul 4: Klassifizierungs- und Regressionsneuronales Netzwerk**

**Name des Studenten: Ihr Name**

# Aufgabenanweisungen

Für diese Aufgabe verwenden Sie den Datensatz **crx.csv**. Dieser Datensatz ist ein öffentlicher Datensatz, den Sie unter [here](https://archive.ics.uci.edu/ml/datasets/credit+approval) finden können. Sie sollten die CSV-Datei auf meiner Datensite an diesem Speicherort verwenden: [here](https://archive.ics.uci.edu/ml/datasets/credit+approval), da sie Spaltenüberschriften enthält. Die Hauptverwendung für diesen Datensatz ist die binäre Klassifizierung. Es gibt 15 Attribute sowie eine Zielspalte, die nur + oder - enthält. In einigen Spalten fehlen Werte.

Sie sollten ein neuronales Netzwerk trainieren und die Vorhersagen zurückgeben. Sie werden diese Vorhersagen an die **Submit**-Funktion übermitteln. Weitere Informationen zum Übermitteln einer Aufgabe oder zum Überprüfen, ob eine Aufgabe übermittelt wurde, finden Sie unter [Assignment #1](https://github.com/jeffheaton/t81_558_deep_learning/blob/master/assignments/assignment_yourname_class1.ipynb).

Führen Sie die folgenden Aufgaben aus:

* Ihre Aufgabe besteht darin, fehlende Werte in den Spalten *a2* und *a14* durch von einem neuronalen Netzwerk geschätzte Werte zu ersetzen (ein neuronales Netzwerk für *a2* und ein anderes für *a14*).
* Ihre Übermittlungsdatei enthält dieselben Kopfzeilen wie die Quell-CSV: *a1*, *a2*, *s3*, *a4*, *a5*, *a6*, *a7*, *a8*, *a9*, *a10*, *a11*, *a12*, *a13*, *a14*, *a15* und *a16*.
* Sie müssen nur *a2* und *a14* ändern.
* Neuronale Netze können beim Ergänzen fehlender Variablen wesentlich leistungsfähiger sein als Median und Mittelwert.
* Trainieren Sie zwei neuronale Netzwerke, um *a2* und *a14* vorherzusagen.
* Das *y* (Ziel) zum Trainieren der beiden Netze ist *a2* und *a14*, je nachdem, welches Sie ausfüllen möchten.
* Die x zum Trainieren der beiden Netze sind 's3','a8','a9','a10','a11','a12','a13','a15'. Diese wurden gewählt, weil es wichtig ist, keine Spalten mit fehlenden Werten zu verwenden; außerdem könnte es zu unerwünschten Verzerrungen führen, wenn wir das endgültige Ziel (*a16*) einbeziehen.
* Sagen Sie NUR neue Werte für fehlende Werte in *a2* und *a14* voraus.
* Sie werden wahrscheinlich diese kleine Warnung erhalten: Warnung: Der Mittelwert der Spalte a14 weicht um 0,20238937709643778 von der Lösungsdatei ab. (ist bei geringer Abweichung möglicherweise egal)



# Google CoLab-Anweisungen

Wenn Sie Google CoLab verwenden, müssen Sie Ihr GDrive mounten, damit Sie Ihr Notebook während des Übermittlungsvorgangs senden können. Wenn Sie den folgenden Code ausführen, wird Ihr GDrive ```/content/drive``` zugeordnet.

In [ ]:
try:
    from google.colab import drive, userdata

    drive.mount("/content/drive", force_remount=True)
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Schlüssel zur Aufgabenabgabe – wurde Ihnen in der ersten Unterrichtswoche zugesandt.
# Wenn Sie in beiden Klassen sind, ist dies derselbe Schlüssel.
if COLAB:
    # Für Colab fügen Sie zu Ihren „Geheimnissen“ hinzu (Schlüsselsymbol links)
    key = userdata.get("T81_558_KEY")
else:
    # Wenn es sich nicht um Colab handelt, geben Sie hier Ihren Schlüssel ein oder verwenden Sie eine Umgebungsvariable.
    # (das ist nur ein Beispielschlüssel, verwenden Sie Ihren)
    key = "Gx5en9cEVvaZnjhdaushddhuhhO4PsI32sgldAXj"

# Frühzeitiges Stoppen (siehe Modul 3.4)
import copy


class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False


# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")

# Funktion zum Übermitteln von Aufgaben

Sie reichen die zehn Programmieraufgaben elektronisch ein. Hierzu können Sie die folgende **Submit**-Funktion verwenden. Mein Server führt eine Grundprüfung jeder Aufgabe durch und informiert Sie, wenn er irgendwelche zugrunde liegenden Probleme erkennt.

**Es ist unwahrscheinlich, dass diese Funktion geändert werden muss.**

In [ ]:
import base64
import os
import numpy as np
import pandas as pd
import requests
import PIL
import PIL.Image
import io
from typing import List, Union

# Mit dieser Funktion können Sie eine Aufgabe einreichen. Sie können eine Aufgabe so oft einreichen, wie Sie möchten, nur die endgültige
# Die Anzahl der Einreichungen. Die Parameter sind wie folgt:
# Daten – Liste von Pandas-Datenrahmen oder -Bildern.
# Schlüssel – Ihr Studentenschlüssel, der Ihnen per E-Mail zugeschickt wurde.
# Kurs – Der Kurs, an dem Sie teilnehmen, derzeit t81-558 oder t81-559.
# nein – Die Zuweisungsklassennummer sollte zwischen 1 und 10 liegen.
# Quelldatei – Der vollständige Pfad zu Ihrer Python- oder IPYNB-Datei. Der Name muss „_class1“ enthalten.
# . Die Nummer muss mit Ihrer Aufgabennummer übereinstimmen. Zum Beispiel „_class2“ für Klassenaufgabe Nr. 2.

def submit(
    data: List[Union[DataFrame, PIL.Image.Image]],
    key: str,
    course: str,
    no: int,
    source_file: str = None
) -> None:
    if source_file is None and '__file__' not in globals():
        raise Exception("Must specify a filename when in a Jupyter notebook.")
    if source_file is None:
        source_file = __file__

    suffix = f'_class{no}'
    if suffix not in source_file:
        raise Exception(f"{suffix} must be part of the filename.")

    ext = os.path.splitext(source_file)[-1].lower()
    if ext not in ['.ipynb', '.py']:
        raise Exception(f"Source file is {ext}; must be .py or .ipynb")

    with open(source_file, "rb") as file:
        encoded_python = base64.b64encode(file.read()).decode('ascii')

    payload = []
    for item in data:
        if isinstance(item, PIL.Image.Image):
            buffered = io.BytesIO()
            item.save(buffered, format="PNG")
            payload.append({'PNG': base64.b64encode(buffered.getvalue()).decode('ascii')})
        elif isinstance(item, DataFrame):
            payload.append({'CSV': base64.b64encode(item.to_csv(index=False).encode('ascii')).decode("ascii")})
        else:
            raise ValueError(f"Unsupported data type: {type(item)}")

    response = requests.post(
        "https://api.heatonresearch.com/wu/submit",
        headers={'x-api-key': key},
        json={
            'payload': payload,
            'assignment': no,
            'course': course,
            'ext': ext,
            'py': encoded_python
        }
    )

    if response.status_code == 200:
print(f"Erfolg: {response.text}")
    else:
print(f"Fehler: {response.text}")

# Aufgabe Nr. 4 Beispielcode

Der folgende Code bietet einen Ausgangspunkt für diese Zuweisung.

In [ ]:
import os
import pandas as pd
from scipy.stats import zscore
import pandas as pd
import io
import requests
import numpy as np
from sklearn import metrics

# Sie müssen Ihre Quelldatei identifizieren. (für Ihr lokales Setup ändern)
file = "/content/drive/My Drive/Colab Notebooks/assignment_yourname_t81_558_class4.ipynb"  # Google CoLab
# Datei = 'C:\\Benutzer\\jeffh\\Projekte\\t81_558_deep_learning\\Aufgaben\\Aufgabe_IhrName_Klasse4.ipynb' # Windows
# Datei = "/Benutzer/jheaton/projects/t81_558_deep_learning/assignments/assignment_IhrName_class4.ipynb" # Mac/Linux

# Aufgabe beginnen
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/crx.csv", na_values=["?"]
)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import tqdm
from sklearn.preprocessing import Labelcoder

hold = None

# # ... fahren Sie mit Ihrem Code fort...

# # Aufgabe einreichen

submit(source_file=file, data=[df_submit], key=key, course="t81-558", no=4)